# Step 3: Econometric Analysis & Visualization

**Paper:** Behavioral Transparency in Multi-Agent RL: Strategic Decoupling and the Economic Optimization of Tactical Shocks  
**Author:** Kenny Ching — School of Business, University of Auckland  

---

### Description
Executes all statistical tests and generates all figures and tables reported in the manuscript.
Evaluates the macroeconomic trajectories of artificial and biological agents following localized
tactical shocks using stratified Nearest-Neighbor state-space matching.

### Reproducibility Targets
| Output | Description |
|---|---|
| **Table 1** | OLS: Strategic Yield after Negative Tactical Shock (with Cohen's d) |
| **Table 2** | OLS: Strategic Yield after Positive Tactical Shock (with Cohen's d) |
| **Fig 1** | KDE: Truncation of Downside Risk |
| **Fig 2** | Bar: Conversion Efficiency (Negative Shock) |
| **Fig 3** | KDE: Maximization of Upside Yield |
| **Fig 4** | Bar: Conversion Efficiency (Positive Shock) |
| **Fig 5** | Econometric Convergence Map |
| **Fig 6** | TOST Equivalence Tests |
| **Fig 7** | Robustness: β_AI across k-matching specifications |

### Input
- `tf_events_master_unified_v2.csv` (from Step 2)

### Key methodological notes
- Standard errors are **HC1 heteroskedasticity-robust** throughout
- Effect sizes reported as **Cohen's d** (pooled SD standardisation)
- TOST equivalence bound = **±500 gold** (~3% of typical late-game net worth)
- k-robustness checks k ∈ {5, 10, 15} nearest neighbours

In [ ]:
# ============================================================
# CELL 1 — Install, import, upload data
# ============================================================
!pip install statsmodels -q

import os
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.neighbors import NearestNeighbors
from scipy.stats import t as t_dist
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files

HORIZONS  = [30, 60, 120, 180]
FIG_DIR   = '/content/figures'
os.makedirs(FIG_DIR, exist_ok=True)

PALETTE = {
    '1. Human Losers':   '#7f7f7f',
    '2. Human Winners':  '#aec7e8',
    '3. Matched Winners':'#1f77b4',
    '4. OpenAI Five':    '#d62728'
}

print('Upload tf_events_master_unified_v2.csv:')
uploaded = files.upload()
csv_path = f"/content/{list(uploaded.keys())[0]}"
with open(csv_path, 'wb') as f:
    f.write(uploaded[list(uploaded.keys())[0]])

df = pd.read_csv(csv_path)
print(f'\nLoaded {len(df):,} rows')
print(f'  AI rows:    {(df["is_ai"]==1).sum()}')
print(f'  Human rows: {(df["is_ai"]==0).sum()}')
print(f'  Shocks: {df["shock_type"].value_counts().to_dict()}')

In [ ]:
# ============================================================
# CELL 2 — Helper functions
# ============================================================
def build_cohorts(df, shock, k=5):
    """Build AI, loser, winner, and matched-winner cohorts for a shock type."""
    sdf        = df[df['shock_type'] == shock].copy()
    ai_ev      = sdf[sdf['is_ai'] == 1].copy()
    hu_losers  = sdf[(sdf['is_ai'] == 0) & (sdf['did_win'] == 0)].copy()
    hu_winners = sdf[(sdf['is_ai'] == 0) & (sdf['did_win'] == 1)].copy()

    features = ['fight_start', 'current_gold_lead']
    scaler   = hu_winners[features].std().replace(0, 1)
    nbrs     = NearestNeighbors(n_neighbors=k).fit(hu_winners[features] / scaler)
    idx      = nbrs.kneighbors(ai_ev[features] / scaler)[1]
    matched  = hu_winners.iloc[np.unique(idx.flatten())].copy()

    ai_ev['Group']     = '4. OpenAI Five'
    hu_losers['Group'] = '1. Human Losers'
    hu_winners['Group']= '2. Human Winners'
    matched['Group']   = '3. Matched Winners'

    return ai_ev, hu_losers, hu_winners, matched

def cohens_d(y, x_binary):
    """Cohen's d with pooled standard deviation."""
    g0 = y[x_binary == 0]
    g1 = y[x_binary == 1]
    n0, n1 = len(g0), len(g1)
    pooled_sd = np.sqrt(((n0-1)*g0.std()**2 + (n1-1)*g1.std()**2) / (n0+n1-2))
    return (g1.mean() - g0.mean()) / pooled_sd if pooled_sd > 0 else 0.0

def tost_pvalue(coef, se, n, bound=500):
    """TOST p-value: max of two one-sided tests against ±bound."""
    df_resid = n - 2
    p_upper  = t_dist.cdf((coef - bound) / se, df=df_resid)
    p_lower  = 1 - t_dist.cdf((coef + bound) / se, df=df_resid)
    return max(p_upper, p_lower)

print('Helper functions defined.')

In [ ]:
# ============================================================
# CELL 3 — Tables 1 & 2: OLS regressions with Cohen's d
# ============================================================
coef_data = []

for shock in ['negative', 'positive']:
    ai_ev, hu_losers, hu_winners, matched = build_cohorts(df, shock, k=5)

    print(f'\n{"="*105}')
    print(f'TABLE: Strategic Yield following a {shock.upper()} Tactical Shock')
    print(f'N — AI: {len(ai_ev)}  Losers: {len(hu_losers)}  Winners: {len(hu_winners)}  Matched: {len(matched)}')
    print(f'{"="*105}')
    print(f'{"Horizon":<7} | {"(1) AI vs Losers  β (d)":<28} | {"(2) AI vs Winners":<25} | {"(3) AI vs Matched"}')
    print('-' * 105)

    for h in HORIZONS:
        col = f'gadv_d{h}'

        d1_data = pd.concat([ai_ev, hu_losers])
        d2_data = pd.concat([ai_ev, hu_winners])
        d3_data = pd.concat([ai_ev, matched])

        m1 = smf.ols(f'{col} ~ is_ai', data=d1_data).fit(cov_type='HC1')
        m2 = smf.ols(f'{col} ~ is_ai', data=d2_data).fit(cov_type='HC1')
        m3 = smf.ols(f'{col} ~ is_ai', data=d3_data).fit(cov_type='HC1')

        c1,p1 = m1.params['is_ai'], m1.pvalues['is_ai']
        c2,p2 = m2.params['is_ai'], m2.pvalues['is_ai']
        c3,p3 = m3.params['is_ai'], m3.pvalues['is_ai']
        d1    = cohens_d(d1_data[col].values, d1_data['is_ai'].values)

        def fmt(c, p):
            sig = '*' if p < 0.05 else 'ns'
            return f'{c:>+8.1f} (p={p:.3f}) {sig}'

        print(f'{h}s     | {fmt(c1,p1)} (d={d1:+.2f}) | {fmt(c2,p2):<25} | {fmt(c3,p3)}')

        for label, m in [('vs Losers',m1),('vs Winners',m2),('vs Matched',m3)]:
            coef_data.append({
                'Shock': shock, 'Horizon': h, 'Model': label,
                'Coef': m.params['is_ai'], 'SE': m.bse['is_ai']
            })

coef_df = pd.DataFrame(coef_data)
print('\n✅ Tables complete.')

In [ ]:
# ============================================================
# CELL 4 — Figures 1 & 2: Negative Shock KDE + Efficiency
# ============================================================
shock = 'negative'
ai_ev, hu_losers, hu_winners, matched = build_cohorts(df, shock)
plot_df = pd.concat([hu_losers, hu_winners, matched, ai_ev]).copy()

# Fig 1: KDE
plt.figure(figsize=(9, 6))
sns.kdeplot(data=plot_df, x='gadv_d180', hue='Group', fill=True,
            common_norm=False, palette=PALETTE, alpha=0.3, linewidth=2)
plt.title('Truncation of Downside Risk (Negative Shock at 180s)', fontsize=14, weight='bold')
plt.xlabel(r'Strategic Yield ($\Delta$ Gold at 180s)')
plt.axvline(0, color='black', ls='--')
plt.xlim(-8000, 8000)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig_negative_distribution.png', dpi=300)
plt.show()

# Fig 2: Conversion Efficiency
plot_df['Success'] = (plot_df['gadv_d180'] > 0).astype(int)
eff = plot_df.groupby('Group')['Success'].agg(['mean','sem']).reset_index()
plt.figure(figsize=(8, 6))
plt.bar(eff['Group'], eff['mean'], yerr=eff['sem']*1.96, capsize=10,
        color=[PALETTE[g] for g in eff['Group']], alpha=0.85, edgecolor='black')
plt.title('Conversion Efficiency: P(Net Surplus) — Negative Shock', fontsize=14, weight='bold')
plt.ylabel(r'P($\Delta$ Gold > 0 at 180s)')
plt.axhline(0.5, color='gray', ls='--', label='Breakeven')
plt.xticks(rotation=15); plt.legend(); plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig_negative_efficiency.png', dpi=300)
plt.show()
print('Figs 1 & 2 saved.')

In [ ]:
# ============================================================
# CELL 5 — Figures 3 & 4: Positive Shock KDE + Efficiency
# ============================================================
shock = 'positive'
ai_ev, hu_losers, hu_winners, matched = build_cohorts(df, shock)
plot_df = pd.concat([hu_losers, hu_winners, matched, ai_ev]).copy()

# Fig 3: KDE
plt.figure(figsize=(9, 6))
sns.kdeplot(data=plot_df, x='gadv_d180', hue='Group', fill=True,
            common_norm=False, palette=PALETTE, alpha=0.3, linewidth=2)
plt.title('Maximization of Upside Yield (Positive Shock at 180s)', fontsize=14, weight='bold')
plt.xlabel(r'Strategic Yield ($\Delta$ Gold at 180s)')
plt.axvline(0, color='black', ls='--')
plt.xlim(-8000, 8000)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig_positive_distribution.png', dpi=300)
plt.show()

# Fig 4: Conversion Efficiency
plot_df['Success'] = (plot_df['gadv_d180'] > 0).astype(int)
eff = plot_df.groupby('Group')['Success'].agg(['mean','sem']).reset_index()
plt.figure(figsize=(8, 6))
plt.bar(eff['Group'], eff['mean'], yerr=eff['sem']*1.96, capsize=10,
        color=[PALETTE[g] for g in eff['Group']], alpha=0.85, edgecolor='black')
plt.title('Conversion Efficiency: P(Net Surplus) — Positive Shock', fontsize=14, weight='bold')
plt.ylabel(r'P($\Delta$ Gold > 0 at 180s)')
plt.axhline(0.5, color='gray', ls='--', label='Breakeven')
plt.xticks(rotation=15); plt.legend(); plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig_positive_efficiency.png', dpi=300)
plt.show()
print('Figs 3 & 4 saved.')

In [ ]:
# ============================================================
# CELL 6 — Figure 5: Econometric Convergence Map
# ============================================================
model_colors = {'vs Losers':'#7f7f7f','vs Winners':'#aec7e8','vs Matched':'#1f77b4'}
model_labels = {
    'vs Losers':  '(1) vs Human Losers',
    'vs Winners': '(2) vs Human Winners',
    'vs Matched': '(3) vs Matched Winners'
}

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
fig.suptitle(
    'Econometric Convergence: AI Advantage Decays with Stricter Biological Baselines',
    fontsize=15, weight='bold'
)
for i, shock in enumerate(['negative','positive']):
    ax = axes[i]
    for model in ['vs Losers','vs Winners','vs Matched']:
        sub = coef_df[(coef_df['Shock']==shock) & (coef_df['Model']==model)].sort_values('Horizon')
        ax.errorbar(sub['Horizon'], sub['Coef'], yerr=sub['SE']*1.96,
                    fmt='-o', label=model_labels[model], color=model_colors[model],
                    capsize=5, linewidth=2.5, markersize=8)
    ax.axhline(0, color='black', ls='--', linewidth=1.5)
    ax.set_title(f'Response to {shock.title()} Shock', fontsize=13)
    ax.set_xlabel('Seconds Post-Shock', fontsize=11)
    ax.set_xticks(HORIZONS)
    ax.grid(True, alpha=0.3)
    if i == 0: ax.set_ylabel(r'AI Marginal Advantage ($\beta_{AI}$ in Gold)', fontsize=11)
    if i == 1: ax.legend(title='Biological Control Group', fontsize=9)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig_econometric_convergence.png', dpi=300)
plt.show()
print('Fig 5 saved.')

In [ ]:
# ============================================================
# CELL 7 — Figure 6: TOST Equivalence Tests
# Equivalence bound = ±500 gold
# ============================================================
TOST_BOUND = 500
tost_results = []

for shock in ['negative','positive']:
    ai_ev, _, _, matched = build_cohorts(df, shock, k=5)
    for h in HORIZONS:
        col      = f'gadv_d{h}'
        combined = pd.concat([ai_ev, matched])
        m        = smf.ols(f'{col} ~ is_ai', data=combined).fit(cov_type='HC1')
        coef, se = m.params['is_ai'], m.bse['is_ai']
        n        = len(combined)
        p        = tost_pvalue(coef, se, n, TOST_BOUND)
        tost_results.append({'Shock':shock,'Horizon':h,'Coef':coef,'p_tost':p})

tost_df = pd.DataFrame(tost_results)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'TOST Equivalence Tests: AI vs Matched Winners (Bound = \u00b5{TOST_BOUND} gold)',
             fontsize=14, weight='bold')
for i, shock in enumerate(['negative','positive']):
    ax  = axes[i]
    sub = tost_df[tost_df['Shock']==shock].sort_values('Horizon')
    bars = ax.bar(sub['Horizon'].astype(str), sub['p_tost'],
                  color='#d62728', alpha=0.7, edgecolor='black', width=0.5)
    ax.axhline(0.05, color='black', ls='--', linewidth=1.5, label='\u03b1 = 0.05')
    ax.set_title(f'{shock.title()} Shock', fontsize=12)
    ax.set_xlabel('Seconds Post-Shock', fontsize=11)
    ax.set_ylabel('p-value (TOST)', fontsize=11)
    ax.set_ylim(0, tost_df['p_tost'].max() * 1.2)
    ax.legend(fontsize=9)
    for bar, (_, row) in zip(bars, sub.iterrows()):
        label = 'Equiv.' if row['p_tost'] < 0.05 else 'n.s.'
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                label, ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig_tost_equivalence.png', dpi=300)
plt.show()
print('Fig 6 saved.')
print('\nTOST results:')
print(tost_df.to_string(index=False))

In [ ]:
# ============================================================
# CELL 8 — Figure 7: k-Robustness Check
# ============================================================
K_VALUES     = [5, 10, 15]
robust_results = []

for k in K_VALUES:
    for shock in ['negative','positive']:
        ai_ev, _, _, matched = build_cohorts(df, shock, k=k)
        combined = pd.concat([ai_ev, matched])
        m        = smf.ols('gadv_d180 ~ is_ai', data=combined).fit(cov_type='HC1')
        coef, se = m.params['is_ai'], m.bse['is_ai']
        n        = len(combined)
        p_tost   = tost_pvalue(coef, se, n, TOST_BOUND)
        robust_results.append({'k':k,'Shock':shock,'Coef':coef,'SE':se,'p_tost':p_tost})

rob_df = pd.DataFrame(robust_results)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle(r'Robustness: $\beta_{AI}$ at 180s Across Matching Specifications (k)',
             fontsize=14, weight='bold')
for i, shock in enumerate(['negative','positive']):
    ax  = axes[i]
    sub = rob_df[rob_df['Shock']==shock].sort_values('k')
    ax.plot(sub['k'], sub['Coef'], '-o', color='#1f77b4', linewidth=2.5, markersize=9)
    ax.fill_between(sub['k'], sub['Coef']-sub['SE']*1.96, sub['Coef']+sub['SE']*1.96,
                    alpha=0.15, color='#1f77b4')
    ax.axhline(0, color='black', ls='--', linewidth=1.5)
    for _, row in sub.iterrows():
        ax.annotate(f"TOST p={row['p_tost']:.3f}",
                    xy=(row['k'], row['Coef']),
                    xytext=(5,8), textcoords='offset points', fontsize=8, color='#1f77b4')
    ax.set_title(f'{shock.title()} Shock', fontsize=12)
    ax.set_xlabel('k (nearest neighbours)', fontsize=11)
    ax.set_ylabel(r'$\beta_{AI}$ (gold at 180s)', fontsize=11)
    ax.set_xticks(K_VALUES)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig_k_robustness.png', dpi=300)
plt.show()
print('Fig 7 saved.')
print('\nRobustness results:')
print(rob_df.to_string(index=False))

In [ ]:
# ============================================================
# CELL 9 — Download all 7 figures as a zip
# ============================================================
import zipfile

zip_path = '/content/figures_all.zip'
with zipfile.ZipFile(zip_path, 'w') as zf:
    for fname in sorted(os.listdir(FIG_DIR)):
        if fname.endswith('.png'):
            zf.write(os.path.join(FIG_DIR, fname), fname)

print('Figures packaged:')
for fname in sorted(os.listdir(FIG_DIR)):
    if fname.endswith('.png'):
        kb = os.path.getsize(os.path.join(FIG_DIR, fname)) // 1024
        print(f'  {fname}  ({kb} KB)')

files.download(zip_path)
print('\n✅ Done — all 7 figures downloaded.')